In [1]:
import pandas as pd
#import matplotlib.ticker as mtick
import numpy as np
import geopandas as gpd
import os

c:\Users\andyli\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\andyli\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
#Layer name for K-12 private enrollment data
lyrK12 = r"E:\xtemp\GitStuff\K-12\TDM-INP-K-12-Enrollment\Private School Universe Survey\Schools_PreKto12_Private.shp"
lyrMAZ = r"E:\xtemp\ABM\PopulationSim\IntermediateFiles1\data-for-andy\microzones_draft_20260610AndyaddedmazidforUtah.shp"

#Column Names
cnLevel    = 'LEVEL2'
cnName     = 'PINST'
cnG00      = 'P160'            #kindergarten as grade 0
cnG01      = 'P180'
cnG02      = 'P200'
cnG03      = 'P210'
cnG04      = 'P220'
cnG05      = 'P230'
cnG06      = 'P240'
cnG07      = 'P250'
cnG08      = 'P260'
cnG09      = 'P270'
cnG10      = 'P280'
cnG11      = 'P290'
cnG12      = 'P300'
cnTot      = 'P305'
cnID       = 'PPIN'

#enrollment levels - grade groups 
elElem = 'Enrol_Elem'
elMidl = 'Enrol_Midl'
elHigh = 'Enrol_High'

#school levels - combo of original data and new levels
slKth8 = 1
slHigh = 2
slOthr = 3

#name of joined layers created later
lyrK12MAZ = 'Schools_PreKto12_Private_withMAZ'
lyrMAZEnrol = 'MAZ_20260625_Enrol'

In [3]:
#grade column names array
cnGrades   = [cnG00,cnG01,cnG02,cnG03,cnG04,cnG05,cnG06,cnG07,cnG08,cnG09,cnG10,cnG11,cnG12]

# list of Enrollment levels by Grade and by School_Level
colnames  =  [cnLevel, cnG00 , cnG01 , cnG02 , cnG03 , cnG04 , cnG05 , cnG06 , cnG07 , cnG08 , cnG09 , cnG10 , cnG11 , cnG12 ]
enroldata = [[slHigh , elHigh, elHigh, elHigh, elHigh, elHigh, elHigh, elHigh, elHigh, elHigh, elHigh, elHigh, elHigh, elHigh],
             [slKth8 , elElem, elElem, elElem, elElem, elElem, elElem, elElem, elMidl, elMidl, elMidl, elMidl, elMidl, elMidl],
             [slOthr , elElem, elElem, elElem, elElem, elElem, elElem, elElem, elMidl, elMidl, elMidl, elHigh, elHigh, elHigh]
            ]
  
# Create the pandas DataFrame
dfEnrollLevels = pd.DataFrame(enroldata, columns = colnames)
print('\nEnrollment Levels by School Level and Grade:')
display(dfEnrollLevels)


Enrollment Levels by School Level and Grade:


,LEVEL2,P160,P180,P200,P210,P220,P230,P240,P250,P260,P270,P280,P290,P300
0,2,Enrol_High,Enrol_High,Enrol_High,Enrol_High,Enrol_High,Enrol_High,Enrol_High,Enrol_High,Enrol_High,Enrol_High,Enrol_High,Enrol_High,Enrol_High
1,1,Enrol_Elem,Enrol_Elem,Enrol_Elem,Enrol_Elem,Enrol_Elem,Enrol_Elem,Enrol_Elem,Enrol_Midl,Enrol_Midl,Enrol_Midl,Enrol_Midl,Enrol_Midl,Enrol_Midl
2,3,Enrol_Elem,Enrol_Elem,Enrol_Elem,Enrol_Elem,Enrol_Elem,Enrol_Elem,Enrol_Elem,Enrol_Midl,Enrol_Midl,Enrol_Midl,Enrol_High,Enrol_High,Enrol_High


In [4]:
dfK12 = gpd.read_file(lyrK12)

#display data totals before filtering
print ("Total Enrollment: " + "{:,}".format(dfK12[cnTot].sum()))
print ("Total Schools: "    + "{:,}".format(dfK12.shape[0])    )

#created filtered dataset that removes unnecessary rows

#make copy to preserve original data
dfK12_fltr = dfK12.copy()

#only school with enrollment
dfK12_fltr = dfK12_fltr[dfK12_fltr[cnTot] > 0]


Total Enrollment: 15,372
Total Schools: 123


In [5]:
#make copy to preserve pre reclassification
dfK12_fltr_rc = dfK12_fltr.copy()
dfK12_fltr_rc[cnLevel] = dfK12_fltr_rc[cnLevel].astype("int32")

#display enrollment by grade to get understanding of the distribution of enrollment for each School_Level
print("\nPre-Reclassify:")
display(dfK12_fltr.groupby([cnLevel], as_index=False).agg(SchoolCount=(cnID,'size'),G00=(cnG00,'sum'),G01=(cnG01,'sum'),G02=(cnG02,'sum'),G03=(cnG03,'sum'),G04=(cnG04,'sum'),G05=(cnG05,'sum'),G06=(cnG06,'sum'),G07=(cnG07,'sum'),G08=(cnG08,'sum'),G09=(cnG09,'sum'),G10=(cnG10,'sum'),G11=(cnG11,'sum'),G12=(cnG12,'sum'),Tot=(cnTot,'sum')))

#display enrollment by grade to get understanding of the distribution of enrollment for each School_Level
print("\nPost-Reclassify:")
display(dfK12_fltr_rc.groupby([cnLevel], as_index=False).agg(SchoolCount=(cnID,'size'),G00=(cnG00,'sum'),G01=(cnG01,'sum'),G02=(cnG02,'sum'),G03=(cnG03,'sum'),G04=(cnG04,'sum'),G05=(cnG05,'sum'),G06=(cnG06,'sum'),G07=(cnG07,'sum'),G08=(cnG08,'sum'),G09=(cnG09,'sum'),G10=(cnG10,'sum'),G11=(cnG11,'sum'),G12=(cnG12,'sum'),Tot=(cnTot,'sum')))


Pre-Reclassify:


,LEVEL2,SchoolCount,G00,G01,G02,G03,G04,G05,G06,G07,G08,G09,G10,G11,G12,Tot
0,1,58,909,4,510,459,420,394,514,431,452,7,1,1,2,7107
1,2,39,1,0,0,2,1,0,7,80,158,636,696,670,588,2840
2,3,26,329,45,338,323,359,358,376,457,431,471,423,398,405,5425



Post-Reclassify:


,LEVEL2,SchoolCount,G00,G01,G02,G03,G04,G05,G06,G07,G08,G09,G10,G11,G12,Tot
0,1,58,909,4,510,459,420,394,514,431,452,7,1,1,2,7107
1,2,39,1,0,0,2,1,0,7,80,158,636,696,670,588,2840
2,3,26,329,45,338,323,359,358,376,457,431,471,423,398,405,5425


In [6]:
#calculate enrollment by level for all schools

#normalize (reverse-pivot) enrollment by grade for ease of calcs
dfK12_melt = pd.melt(dfK12_fltr_rc, id_vars=[cnID,cnLevel], value_vars=cnGrades)
dfK12_melt = dfK12_melt.rename(columns={'variable':'Grade','value':'Enrollment'})
#display(dfK12_melt)

#create 'lookup table' for Enrollment Level
dfEnrollLevels_melt = pd.melt(dfEnrollLevels, id_vars=[cnLevel], value_vars=cnGrades)
dfEnrollLevels_melt = dfEnrollLevels_melt.rename(columns={'variable':'Grade','value':'EnrollmentLevel'})
#display(dfEnrollLevels_melt)

#join to get enrollment levels
dfK12_enrolllevels = pd.DataFrame.merge(dfK12_melt,dfEnrollLevels_melt,on=(cnLevel,'Grade'))

#group to get subtotals by School, Group, and Enrollment Level
dfK12_enrolllevels = dfK12_enrolllevels.groupby([cnID,'EnrollmentLevel'],as_index=False).agg(Enrollment=('Enrollment','sum'))
dfK12_enrolllevels

,PPIN,EnrollmentLevel,Enrollment
0,1412727,Enrol_Elem,150
1,1412727,Enrol_Midl,45
2,1412738,Enrol_Elem,68
3,1412738,Enrol_Midl,23
4,1412749,Enrol_Elem,103
...,...,...,...
228,AA890947,Enrol_Elem,6
229,AA890947,Enrol_High,26
230,AA890947,Enrol_Midl,13
231,K9306116,Enrol_Elem,54


In [7]:
# Pivot by EnrollmentLevel
dfK12_enrol = dfK12_enrolllevels.pivot(index=cnID, columns='EnrollmentLevel', values='Enrollment')

# Flatten the column names (though it might not be needed anymore with single-level columns)
dfK12_enrol.columns = [str(col) for col in dfK12_enrol.columns]

# Fill missing values and convert to integers
dfK12_enrol = dfK12_enrol.fillna(0).astype(int)

display(dfK12_enrol)
display(dfK12_enrol.sum())
display(dfK12_enrol.sum().sum())

,Enrol_Elem,Enrol_High,Enrol_Midl
PPIN,,,
1412727,150,0,45
1412738,68,0,23
1412749,103,0,48
1412771,0,504,0
1412818,0,171,0
...,...,...,...
A9904321,292,0,89
A9904322,62,0,16
AA001502,394,234,264


Enrol_Elem    5338
Enrol_High    4065
Enrol_Midl    2253
dtype: int64

11656

In [8]:
#export school enrollment results to csv
strprivK12File = "results\privk12_enrol.csv"

dfK12_enrol.to_csv(strprivK12File, index=False)

# =========================================================
# 2. ATTRIBUTE JOIN (Equivalent to AddJoin 'KEEP_ALL')
# =========================================================
# how='left' acts as 'KEEP_ALL', retaining all school points.
gdf_k12_joined = dfK12.merge(dfK12_enrol, on=cnID, how='left')

gdf_maz= gpd.read_file(lyrMAZ)

# =========================================================
# 3. SPATIAL JOIN (Equivalent to SpatialJoin 'KEEP_COMMON')
# =========================================================
# Best practice: Always align Coordinate Reference Systems (CRS) first!
if gdf_k12_joined.crs != gdf_maz.crs:
    print(f"Aligning CRS to match MAZs ({gdf_maz.crs})...")
    gdf_k12_joined = gdf_k12_joined.to_crs(gdf_maz.crs)

# how='inner' replicates 'KEEP_COMMON'
# predicate='intersects' is used here instead of 'within' to prevent boundary-dropping bugs
gdf_k12_maz = gpd.sjoin(gdf_k12_joined, gdf_maz, how='inner', predicate='intersects')

# =========================================================
# 4. REMOVE JOIN
# =========================================================
# NOT NEEDED! 
# Your original `gdf_k12` variable remains completely untouched in memory.

Aligning CRS to match MAZs (EPSG:26912)...


In [9]:
#get joined school/TAZ layer in order to aggregate to CO_TAZID
sdfK12MAZ = gdf_k12_maz.copy()
dfK12MAZ = sdfK12MAZ[[cnName,'maz_id',elElem,elMidl,elHigh]]
#dfK12TAZ = sdfK12TAZ.fillna(0)
#display(dfK12TAZ.columns)
display(dfK12MAZ)

,PINST,maz_id,Enrol_Elem,Enrol_Midl,Enrol_High
0,ST VINCENT DE PAUL PARISH SCHOOL,11001,150,45,0
1,KEARNS-SAINT ANN SCHOOL,12420,68,23,0
2,OUR LADY OF LOURDES CATHOLIC SCHOOL,12800,103,48,0
3,JUDGE MEMORIAL CATHOLIC HIGH SCHOOL,12800,0,0,504
4,ST JOSEPH CATHOLIC HIGH SCHOOL,19690,0,0,171
...,...,...,...,...,...
117,THE MADELEINE CHOIR SCHOOL,12983,292,89,0
118,PRINCE OF PEACE EVANGELICAL LUTHERAN SCHOOL,11171,62,16,0
119,THE WATERFORD SCHOOL,8690,394,264,234
120,UNIVERSITY ACADEMY,21811,6,13,26


In [10]:
#aggregate by CO_TAZID
dfMAZSummary = dfK12MAZ.groupby(['maz_id']).agg(sum)
dfMAZSummary[dfMAZSummary.select_dtypes(include='number').columns] = \
    dfMAZSummary.select_dtypes(include='number').astype(int)

#remove rows with all zeros
dfMAZSummary = dfMAZSummary.loc[~(dfMAZSummary==0).all(axis=1)]

dfMAZSummary

C:\Users\andyli\AppData\Local\Temp\ipykernel_44492\759562257.py:2: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  dfMAZSummary = dfK12MAZ.groupby(['maz_id']).agg(sum)


,PINST,Enrol_Elem,Enrol_Midl,Enrol_High
maz_id,,,,
7120,HOLY TRINITY LUTHERAN SCHOOL,19,0,0
7572,ST ANDREW CATHOLIC SCHOOL,86,21,0
7595,ST JOHN THE BAPTIST MIDDLE SCHOOL,96,237,0
7681,GATEWAY ACADEMY,0,0,1
7790,BELL CANYON MONTESSORI SCHOOL,13,0,0
...,...,...,...,...
31228,PROVO CANYON SCHOOL-PROVO CAMPUS,10,9,0
31836,DISCOVERY ACADEMY,0,0,62
32140,THE HERRITAGE SCHOOL,0,0,111


In [11]:
#export CO_TAZID enrollment to csv

sFilename = r'results\privK12_Enrollment_maz.csv'

dfMAZSummary.to_csv(sFilename)
print('CSV Exported to: ' + sFilename)

CSV Exported to: results\privK12_Enrollment_maz.csv


In [12]:
import pandas as pd
import geopandas as gpd

# Assuming 'lyrMAZ' is the file path to your MAZ shapefile.
# (If you already have it loaded as a GeoDataFrame from earlier in your script, 
# you can skip the read_file line and just use that variable!)



# =========================================================
# 2. ATTRIBUTE JOIN (Equivalent to AddJoin 'KEEP_ALL')
# =========================================================
# how='left' acts exactly like 'KEEP_ALL', keeping all MAZs even if they have no private schools
print("Executing attribute join...")
gdf_maz_joined = gdf_maz.merge(dfMAZSummary, on='maz_id', how='left')

# =========================================================
# 3. EXPORT TO SHAPEFILE (Equivalent to CopyFeatures)
# =========================================================
print("Exporting joined data to shapefile...")
output_shapefile = r"results\privtaz_with_enrollment.shp"
gdf_maz_joined.to_file(output_shapefile)

# =========================================================
# 4. REMOVE JOIN
# =========================================================
# NOT NEEDED! 
# Your original `gdf_maz` variable remains completely untouched. 
print(f"Success! Exported to {output_shapefile}")

Executing attribute join...
Exporting joined data to shapefile...
Success! Exported to results\privtaz_with_enrollment.shp
